In [3]:
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [4]:
from langchain.tools import tool

Part B — Coding Exercises
All exercises use GreenPlate, a restaurant reservation and food ordering assistant. Each question gives you a blank code cell to work in — write your solution there and run it to confirm it works before moving on.

B1. Easy
B1.1 — A basic tool. Write a @tool-decorated function called check_table_availability that takes a party_size: int and a time_slot: str, and returns a string saying whether a table is available (you can hardcode fake availability data, e.g. tables available for parties of 2-6 at "7:00 PM" and "8:30 PM" only). Include a proper docstring. Print the tool's .name, .description, and .args to confirm it's built correctly.

In [7]:
@tool
def check_table_availability(party_size: int, time_slot: str) -> str:
    """check for table available for a given party size and timeslot"""
    if party_size < 10:
        return f"For {party_size} no booking needed"
    elif 10 <= party_size <= 20:
        return f"For {party_size} available for 2:00 - 4:00 PM"
    else:
        return f"For {party_size} available after 9:00 PM"


In [8]:
print(check_table_availability.name)
print(check_table_availability.description)
print(check_table_availability.args)

check_table_availability
check for table available for a given party size and timeslot
{'party_size': {'title': 'Party Size', 'type': 'integer'}, 'time_slot': {'title': 'Time Slot', 'type': 'string'}}


B1.2 — A reusable prompt template. Build a ChatPromptTemplate that generates a short, enthusiastic description of a dish, given {dish_name} and {cuisine_type} as variables. Run it with at least two different dish/cuisine combinations and print both results.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

template = ChatPromptTemplate.from_messages(
    [
        ("system","You are an assistant who generates a short, enthusiastic description of a dish based on a cuisine. "),
        ("user", "Tell me about {dish_name} which is a {cuisine_type}  cuisine. ")
    ]
)

model = init_chat_model("openai:gpt-5-mini")
chain = template | model

for dish,cuisine in [("paneer tikka masala", "indian"), ("pizza", "italian")]:
    result = chain.invoke({"dish_name": dish,"cuisine_type": cuisine})
    print(result.content)

Paneer tikka masala is a vibrant, indulgent North Indian dish that pairs cubes of paneer (fresh Indian cottage cheese) marinated in spiced yogurt, grilled or roasted until slightly charred, then simmered in a rich, creamy tomato-onion masala. It balances smoky, tangy and mildly spicy flavors—think garam masala, cumin, coriander, turmeric and a touch of chili—rounded out with butter or cream for a silky finish. Bright cilantro and a squeeze of lemon lift the dish, which is classically served with naan, roti or basmati rice. Vegetarian, comforting, and restaurant-favorite deliciousness in every bite.
Pizza is the joyful heart of Italian cuisine: a blistered, chewy crust topped with tangy San Marzano tomatoes, creamy mozzarella and a scatter of fresh basil, baked in a roaring wood-fired oven until the edges are charred and the cheese bubbles. Simple yet endlessly adaptable, from classic Neapolitan Margherita to crispy Roman slices or rich, rustic toppings, each bite balances vivid flavors

B1.3 — A schema with a constrained field. Define a Pydantic BaseModel called FoodOrder with fields: customer_name (str), dish_name (str), quantity (int, must be between 1 and 10), and spice_level (a Literal restricted to "mild", "medium", or "hot"). Use with_structured_output() to extract a FoodOrder from this message: "Hi, I'm Karan, 2 butter chicken please, medium spice." Print the result.

In [15]:
from pydantic import BaseModel,Field
from typing import Literal 
from langchain.chat_models import init_chat_model
class FoodOrder(BaseModel):
    customer_name: str = Field(description="Customer name")
    dish_name: str = Field(description="Dish name")
    quantity: int = Field(description="Quantity" , gt=1, lt=10)
    spice_level: Literal["mild", "medium", "hot"]

model = init_chat_model("openai:gpt-5-mini")
model_with_structured_output = model.with_structured_output(FoodOrder)
result = model_with_structured_output.invoke("Extract FoodOrder message for this - \
Hi, I'm Karan, 2 butter chicken please, medium spice.")
print(result)

result = model_with_structured_output.invoke("Extract FoodOrder message for this - \
Hi, I'm Karan, 11 butter chicken please, medium spice.")
print(result)

customer_name='Karan' dish_name='butter chicken' quantity=2 spice_level='medium'
customer_name='Karan' dish_name='butter chicken' quantity=9 spice_level='medium'


B2. Medium
B2.1 — Two schemas, one agent. A restaurant assistant needs to handle both new reservations and cancellations. Define NewReservation (customer_name, party_size, time_slot) and CancelReservation (customer_name, time_slot) as separate Pydantic models. Build a create_agent with response_format=ToolStrategy(Union[NewReservation, CancelReservation]), and test it with one clearly-a-reservation message and one clearly-a-cancellation message. Use isinstance() to print which schema was chosen each time.

In [29]:
from pydantic import BaseModel,Field
from typing import Literal,Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class NewReservation(BaseModel):
    customer_name: str = Field(description="customer name")
    party_size: int = Field(description="party size")
    time_slot: str = Field(description="time slot")

class CancelReservation(BaseModel):
    customer_name: str = Field(description="customer name")
    time_slot: str = Field(description="time slot")

agent = create_agent(
    model="openai:gpt-5-mini",
    response_format = ToolStrategy(Union[NewReservation,CancelReservation])
)
result = agent.invoke({
    "messages": [
      {
          "role": "user",
          "content": "Create a new reserveration for Rohan for a party size of 10 between 8-10pm"
      }
    ]
})

print(result["structured_response"], type(result["structured_response"]))


result = agent.invoke({
    "messages": [
      {
          "role": "user",
          "content": "Update my reserveration for Rohan from  party size 10 to 20 between 8-10pm"
      }
    ]
})

print(result["structured_response"], type(result["structured_response"]))

customer_name='Rohan' party_size=10 time_slot='8-10pm' <class '__main__.NewReservation'>
customer_name='Rohan' time_slot='8:00-10:00 PM' <class '__main__.CancelReservation'>
